# GTAA Multi-Agent System — Interactive Walkthrough

This notebook demonstrates the full agent pipeline on live market data.
Run each cell in sequence to see how the 8 agents collaborate.

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from config.settings import GTAAConfig, ASSET_UNIVERSE
from data.data_loader import DataLoader

# Load market data
config = GTAAConfig()
loader = DataLoader(universe=config.universe, start='2023-01-01')
prices = loader.load()
returns = loader.returns

print(f'Loaded {len(prices.columns)} assets, {len(prices)} trading days')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Assets: {list(prices.columns)}')

In [ ]:
import yfinance as yf

# Load VIX for regime classification
vix_raw = yf.download('^VIX', start='2023-01-01', progress=False)
if isinstance(vix_raw.columns, pd.MultiIndex):
    vix_series = vix_raw['Close']['^VIX'].reindex(prices.index).ffill()
else:
    vix_series = vix_raw['Close'].reindex(prices.index).ffill()

print(f'VIX current: {vix_series.iloc[-1]:.1f}')
print(f'VIX 20d mean: {vix_series.iloc[-20:].mean():.1f}')

## Step 1 — Research Agent: Momentum Scan

In [ ]:
from agents.research_agent import ResearchAgent

date = prices.index[-1]
research = ResearchAgent()
r_sig = research.analyze(date, prices, returns, {'universe': ASSET_UNIVERSE})

print(f'Date: {date.date()}')
print(f'Confidence: {r_sig.confidence:.2f}')
print(f'Reasoning: {r_sig.reasoning}')
print(f'\nTop 5 Momentum: {r_sig.data["top_5"]}')
print(f'Bottom 5: {r_sig.data["bottom_5"]}')
print(f'% Above 200d SMA: {r_sig.data["uptrend_pct"]:.0%}')

## Step 2 — ML Regime Agent: XGBoost 5-Regime Classification

In [ ]:
from agents.ml_regime_agent import MLRegimeAgent

ml_regime = MLRegimeAgent(retrain_every=9999)
rg_sig = ml_regime.analyze(date, prices, returns, {'vix_series': vix_series})

print(f'Regime: {rg_sig.data["regime"]}')
print(f'Action: {rg_sig.data["action"]}')
print(f'Confidence: {rg_sig.confidence:.2f}')
print(f'Bull Ratio: {rg_sig.data["bull_ratio"]:.2f}')
print(f'ML Trained: {rg_sig.data["ml_trained"]}')
print(f'\nProbabilities:')
for regime, prob in rg_sig.data.get('probabilities', {}).items():
    bar = '█' * int(prob * 50)
    print(f'  {regime:>12}: {prob:.1%} {bar}')

## Step 3 — ML Direction Agent: RF + KNN Per-Asset Ensemble

In [ ]:
from agents.ml_direction_agent import MLDirectionAgent

ml_dir = MLDirectionAgent(retrain_every=9999)
dir_sig = ml_dir.analyze(date, prices, returns, {})

print(f'Models trained: {dir_sig.data["n_models_trained"]}')
print(f'\nBullish: {dir_sig.data["bullish_tickers"]}')
print(f'Bearish: {dir_sig.data["bearish_tickers"]}')
print(f'\nTop predictions:')
preds = dir_sig.data['predictions']
for t, p in sorted(preds.items(), key=lambda x: -x[1]['score'])[:8]:
    direction = '↑' if p['direction'] > 0 else '↓'
    print(f'  {t:>10} {direction} conf={p["confidence"]:.2f} score={p["score"]:+.3f}')

## Step 4 — Risk Agent: Limit Enforcement

In [ ]:
from agents.risk_agent import RiskAgent

# Build proposed weights from top momentum
top_tickers = r_sig.data['top_5'][:5]
proposed = {t: round(0.25 - i*0.04, 2) for i, t in enumerate(top_tickers)}
print(f'Proposed weights: {proposed}')

risk = RiskAgent(config=config.risk)
risk_sig = risk.analyze(date, prices, returns, {
    'proposed_weights': proposed,
    'universe': ASSET_UNIVERSE,
    'equity_value': 100000,
    'regime': rg_sig.data.get('regime_3way', 'RISK_ON'),
})

print(f'\nVetoed: {risk_sig.data["vetoed"]}')
print(f'Approved weights: {risk_sig.data["approved_weights"]}')
print(f'Violations: {risk_sig.data["violations"]}')
print(f'Drawdown level: {risk_sig.data["drawdown_level"]:.1%}')

## Full Signal Audit Trail

Every signal is a typed Python dataclass that can be serialized to JSON for audit.

In [ ]:
# Show the raw Signal object structure
print('=== Research Signal ===')
print(json.dumps(r_sig.to_dict(), indent=2, default=str)[:500])
print('...')
print(f'\n=== Regime Signal ===')
print(json.dumps(rg_sig.to_dict(), indent=2, default=str)[:500])
print('...')

## Run Full Backtest

To run the complete V5 backtest that produced the README metrics:

In [ ]:
# Uncomment to run (takes ~30 seconds):
# !python3 run_backtest.py

## Run Monte Carlo Simulation

In [ ]:
# Uncomment to run (takes ~3 seconds):
# !python3 -m engine.monte_carlo